# Xoá lịch sử chat của một khách hàng trên server đã deploy

Notebook đứng riêng, **không sửa gì trong repo** -- chỉ gọi qua HTTP tới server đã deploy trên GreenNode AgentBase Runtime.

**Vì sao cần notebook riêng thay vì gọi thẳng `DELETE /chat/history`:** endpoint đó hiện luôn trả lỗi `500` cho MỌI khách hàng. Đọc code
(`server.py`) cho thấy lịch sử chat của server hiện đang nằm trong RAM của tiến trình (`MemorySaver()`, từ commit `46d99e2`) chứ không còn
nằm trong GreenNode AgentBase Memory như thiết kế gốc (`AgentBaseMemoryEvents(memory_id=MEMORY_ID)`, bị bỏ ở cùng commit đó và chưa được
khôi phục ở commit sau). `DELETE /chat/history` vẫn gọi `checkpointer.delete_thread(thread_id, user_id)` (2 tham số) -- đúng chữ ký của
`AgentBaseMemoryEvents`, nhưng `MemorySaver` hiện tại chỉ nhận 1 tham số nên luôn `TypeError` -> `500`.

**Notebook này làm 2 việc, và nói rõ việc nào thật sự xoá được lịch sử khách đang thấy:**

1. Thử gọi đúng `DELETE /chat/history` qua HTTP (không sửa code) -- nếu server đã được vá/redeploy thì việc này tự xoá được, cell in kết quả thật.
2. Dọn trực tiếp các event của khách trong GreenNode AgentBase Memory resource (`MEMORY_ID`) -- việc này luôn làm được và không hại gì, nhưng
   **theo code hiện tại sẽ KHÔNG làm thay đổi những gì `GET /chat/history` trả về**, vì server không đọc từ resource đó nữa. Notebook in rõ
   kết quả `GET /chat/history` trước và sau để bạn tự thấy đúng như vậy, thay vì tôi khẳng định suông.

Muốn xoá thật sự những gì khách đang thấy thì cần sửa `server.py` (khôi phục `AgentBaseMemoryEvents`, hoặc ít nhất sửa `DELETE /chat/history`
cho khớp chữ ký `MemorySaver`) rồi redeploy -- việc đó đụng vào code nên không nằm trong notebook này, chờ bạn quyết.

Cần trong `.env` (repo root): `SERVER_API_KEY`, `MEMORY_ID`, và 4 biến Auth0 (`AUTH0_DOMAIN`, `AUTH0_CLIENT_ID`, `AUTH0_CLIENT_SECRET`,
`AUTH0_AUDIENCE`) -- lấy từ người quản lý Auth0 (xem `docs/api.md`, lớp 2).

In [ ]:
import os
import sys
from pathlib import Path

import httpx
from dotenv import load_dotenv

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
load_dotenv(REPO_ROOT / ".env")

# --- đổi 2 dòng này cho khách hàng và server cần xử lý ---
CUSTOMER_ID = "CIF_0002"
SERVER_URL = "https://endpoint-e14e9088-1840-467b-8e72-4cdad04e0c93.agentbase-runtime.aiplatform.vngcloud.vn"

REQUIRED = ("SERVER_API_KEY", "MEMORY_ID", "AUTH0_DOMAIN", "AUTH0_CLIENT_ID", "AUTH0_CLIENT_SECRET", "AUTH0_AUDIENCE")
missing = [name for name in REQUIRED if not os.environ.get(name)]
assert not missing, f"Thiếu biến trong {REPO_ROOT / '.env'}: {missing}"

token_response = httpx.post(
    f"https://{os.environ['AUTH0_DOMAIN']}/oauth/token",
    json={
        "client_id": os.environ["AUTH0_CLIENT_ID"],
        "client_secret": os.environ["AUTH0_CLIENT_SECRET"],
        "audience": os.environ["AUTH0_AUDIENCE"],
        "grant_type": "client_credentials",
    },
    timeout=30,
)
token_response.raise_for_status()
JWT = token_response.json()["access_token"]
HEADERS = {"X-API-Key": os.environ["SERVER_API_KEY"], "Authorization": f"Bearer {JWT}"}
print("Auth0 token OK, server:", SERVER_URL, "| khách:", CUSTOMER_ID)

In [ ]:
# 1. Xem trước lịch sử hiện tại của khách (để biết sẽ mất bao nhiêu tin nhắn)
response = httpx.get(f"{SERVER_URL}/chat/history", headers=HEADERS, params={"user_id": CUSTOMER_ID}, timeout=60)
response.raise_for_status()
before = response.json()
print(f"thread_id: {before['thread_id']}  |  số tin nhắn: {len(before['messages'])}")
for m in before["messages"][:3] + before["messages"][-2:]:
    print(f"  {m['role']:<9} | {m['content'][:90]}")

In [ ]:
# 2. Thử xoá bằng đúng endpoint DELETE /chat/history (không sửa code -- gọi y như frontend sẽ gọi)
response = httpx.delete(f"{SERVER_URL}/chat/history", headers=HEADERS, params={"user_id": CUSTOMER_ID}, timeout=60)
print("HTTP", response.status_code, "|", response.text)

if response.status_code == 200:
    print("\nXoá thành công qua DELETE /chat/history.")
elif response.status_code == 500:
    print(
        "\n500 như dự đoán: bug đã mô tả ở đầu notebook (checkpointer.delete_thread nhận sai số tham số).\n"
        "Cell dưới sẽ dọn trực tiếp trong GreenNode AgentBase Memory, nhưng lưu ý: theo code hiện tại,\n"
        "việc đó sẽ KHÔNG xoá được những gì GET /chat/history đang trả (xem cell kiểm tra cuối)."
    )
else:
    print("\nMã lỗi không như dự đoán -- đọc kỹ thông báo ở trên trước khi làm tiếp.")

In [ ]:
# 3. Dọn trực tiếp trong GreenNode AgentBase Memory resource (MEMORY_ID) -- luôn an toàn để chạy,
# nhưng KHÔNG ảnh hưởng tới GET /chat/history hiện tại (xem lời giải thích ở đầu notebook).
from greennode_agent_bridge import AgentBaseMemoryEvents

checkpointer = AgentBaseMemoryEvents(memory_id=os.environ["MEMORY_ID"])
memory_client = checkpointer.checkpoint_event_client.memory_client
memory_id = os.environ["MEMORY_ID"]

deleted = 0
while True:
    page = memory_client.list_events(id=memory_id, actorId=CUSTOMER_ID, sessionId=CUSTOMER_ID, page=1, size=100)
    events = page.list_data or []
    if not events:
        break
    for event in events:
        if event.id:
            memory_client.delete_event(id=memory_id, actorId=CUSTOMER_ID, sessionId=CUSTOMER_ID, eventId=event.id)
            deleted += 1
print(f"Đã xoá {deleted} event của {CUSTOMER_ID} trong Memory resource {memory_id[-6:]}...")

In [ ]:
# 4. Kiểm tra lại -- in rõ để thấy GET /chat/history có đổi hay không sau bước 3
response = httpx.get(f"{SERVER_URL}/chat/history", headers=HEADERS, params={"user_id": CUSTOMER_ID}, timeout=60)
response.raise_for_status()
after = response.json()
print(f"Trước: {len(before['messages'])} tin nhắn  |  Sau khi dọn Memory resource: {len(after['messages'])} tin nhắn")
if len(after["messages"]) == len(before["messages"]):
    print(
        "\nKhông đổi -- đúng như dự đoán: server đang đọc lịch sử từ RAM (MemorySaver), không phải từ Memory resource.\n"
        "Muốn CIF này thật sự hết lịch sử thì cần sửa server.py và redeploy (mục 'Sửa hướng nào' đã hỏi bạn), hoặc\n"
        "restart/redeploy container đang chạy (nhưng việc đó xoá RAM của TẤT CẢ khách hàng, không riêng CIF này)."
    )
else:
    print("\nCó đổi -- server có vẻ đang thực sự đọc từ Memory resource (khác với những gì code hiện đọc được).")